In [19]:
import numpy as np
import pandas as pd

temp_df = pd.read_csv("IMDB Dataset.csv")

df = temp_df.iloc[:10000]
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [20]:
df['review'][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [21]:
df['sentiment'].value_counts()

,count
sentiment,
positive,5028
negative,4972


In [22]:
df.isnull().sum()

,0
review,0
sentiment,0


In [23]:
df.duplicated().sum()

np.int64(17)

In [24]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_1289/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [25]:
df.duplicated().sum()

np.int64(0)

In [26]:
# Basic Preprocessing
# Remove Tags
# Lowercase
# Remove stopwords

In [27]:
import re
def remove_tags(raw_text):
  cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
  return cleaned_text

In [28]:
df['review'] = df['review'].apply(remove_tags)

/tmp/ipykernel_1289/2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [29]:
df['review'] = df['review'].apply(lambda x: x.lower())

/tmp/ipykernel_1289/2432328559.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: x.lower())


In [30]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [31]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x: " ".join(x))

/tmp/ipykernel_1289/2139393212.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x: " ".join(x))


In [32]:
df.head(5)

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive


In [33]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [34]:
X

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
9995,"fun, entertaining movie wwii german spy (julie..."
9996,"give break. anyone say ""good hockey movie""? kn..."
9997,movie bad movie. watching endless series bad h...
9998,"movie probably made entertain middle school, e..."


In [35]:
y

,sentiment
0,positive
1,positive
2,positive
3,negative
4,positive
...,...
9995,positive
9996,negative
9997,negative
9998,negative


In [36]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [37]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [38]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
X_train.shape

(7986, 1)

In [40]:
# Applying BOW

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()

In [41]:
X_train_bow.shape

(7986, 48247)

In [42]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow, y_train)

GaussianNB()

In [43]:
y_pred = gnb.predict(X_test_bow)

from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(y_test, y_pred)

0.627941912869304

In [44]:
confusion_matrix(y_test, y_pred)

array([[694, 291],
       [452, 560]])

In [45]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.8552829243865798

In [46]:
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.transform(X_test['review']).toarray()


rf = RandomForestClassifier()

rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.8452679018527791

##Using TFIDF

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [48]:
X_train_tfidf = tfidf.fit_transform(X_train['review']).toarray()
X_test_tfidf = tfidf.transform(X_test['review'])

In [49]:
rf = RandomForestClassifier()

rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)

accuracy_score(y_test, y_pred)

0.8537806710065098

##WORD2VEC

In [50]:
import numpy as np
import pandas as pd
df = pd.read_csv("IMDB Dataset.csv")

In [51]:
df.drop_duplicates(inplace=True)

In [52]:
import re
def remove_tags(raw_text):
  cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
  return cleaned_text

In [53]:
df['review'] = df['review'].apply(remove_tags)

In [54]:
df['review'] = df['review'].apply(lambda x: x.lower())

In [55]:
df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x: " ".join(x))

In [56]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 83.0 MB/s eta 0:00:00


In [57]:
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [58]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [59]:
story = []

for doc in df['review']:
  raw_sent = sent_tokenize(doc)
  for sent in raw_sent:
    story.append(simple_preprocess(sent))

In [60]:
model = gensim.models.Word2Vec(
    window=10,
    min_count = 2
)

In [61]:
model.build_vocab(story)

In [62]:
model.train(story, total_examples=model.corpus_count, epochs=model.epochs)

(29413297, 30767745)

In [63]:
len(model.wv.index_to_key)

61843

In [64]:
def document_vector(doc):
  doc = [word for word in doc.split() if word in model.wv.index_to_key]
  return np.mean(model.wv[doc], axis=0)

In [65]:
document_vector(df['review'].values[0])

array([ 0.08546538, -0.09242506,  0.0498133 ,  0.11101915,  0.66815954,
       -0.46389285,  0.16725048,  0.51239127,  0.5257404 , -0.23138484,
       -0.08113548, -0.35603747, -0.21783014,  0.32249612,  0.39639264,
        0.12817585, -0.01612317, -0.41946325, -0.188402  , -0.33557695,
        0.09027708,  0.27584746, -0.13488893,  0.2623219 , -0.38244817,
       -0.0030819 , -0.2666436 ,  0.01166236, -0.29118025, -0.26507238,
        0.18308339, -0.24185394,  0.01334761, -0.40736082,  0.02955475,
        0.49304456, -0.4157883 ,  0.06200635,  0.14445488, -0.44070154,
       -0.34774938,  0.15024105, -0.23587865, -0.3351919 ,  0.14897291,
       -0.08754054,  0.08305283, -0.06399363, -0.243819  ,  0.41676995,
        0.3487848 , -0.0922371 , -0.15328915, -0.34906694, -0.24657546,
        0.28861752, -0.09218102,  0.30399993, -0.26503503,  0.25845107,
        0.16664484, -0.4676043 , -0.05453994,  0.3443193 , -0.13597736,
        0.13367724, -0.22795326, -0.03315249,  0.19642825, -0.12

In [ ]:
from tqdm import tqdm

x = []
for doc in tqdm(df['review'].values):
  x.append(document_vector(doc))

  8%|▊         | 3779/49582 [01:50<28:48, 26.51it/s]

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])
y

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)